# 12. Bias-Variance Tradeoff

**Statistical Foundations for Data Science — Notebook 12 of 12**

Notebook 11 showed *that* models overfit and how to treat it. This notebook explains *why*
the trade-off exists at all — and it turns out to be an identity, not a heuristic. Prediction
error decomposes exactly into three pieces, two of which you control and one of which you
cannot.

Once you can see the decomposition, every modelling decision becomes legible: regularisation
is buying bias to sell variance, bagging is variance reduction, boosting is bias reduction.

### What you will learn

1. The formal **bias-variance decomposition** and its derivation
2. **Estimating** bias and variance by simulation, since we can only do it when we know the truth
3. Why bias and variance move in opposite directions as complexity grows
4. **Irreducible error** — the floor no model can beat
5. Where common models sit on the spectrum
6. How each remedy (regularisation, bagging, boosting, more data, more features) moves the
   three terms
7. The **classification** version of the trade-off
8. **Double descent** — where the classical picture breaks down

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (RandomForestRegressor, BaggingRegressor,
                              GradientBoostingRegressor)
from sklearn.neighbors import KNeighborsRegressor

rng = np.random.default_rng(seed=12)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)

---
## 12.1 The decomposition

Suppose the truth is $y = f(x) + \varepsilon$ with $E[\varepsilon]=0$ and
$\operatorname{Var}(\varepsilon) = \sigma^2$. You fit $\hat{f}$ on a random training set.
The **expected squared error at a fixed point $x$**, averaged over all possible training
sets, is

$$\mathbb{E}\big[(y - \hat{f}(x))^2\big]
= \underbrace{\big(\mathbb{E}[\hat{f}(x)] - f(x)\big)^2}_{\textbf{Bias}^2}
+ \underbrace{\mathbb{E}\big[(\hat{f}(x) - \mathbb{E}[\hat{f}(x)])^2\big]}_{\textbf{Variance}}
+ \underbrace{\sigma^2}_{\textbf{Irreducible}}$$

In words:

- **Bias** — how far the *average* model is from the truth. Systematic error from wrong
  assumptions: a straight line fitted to a curve is biased everywhere.
- **Variance** — how much the model changes if you hand it a different training sample.
  Sensitivity to the particular data you happened to get.
- **Irreducible error** ($\sigma^2$) — noise in $y$ itself. No model, however good, can go
  below it. If someone reports beating it, they have leaked something.

### The darts analogy

Bias is aiming at the wrong spot. Variance is a shaky hand. You can have

| | Low variance | High variance |
|---|---|---|
| **Low bias** | tight cluster on the bullseye ✅ | scattered around the bullseye |
| **High bias** | tight cluster in the wrong place | scattered *and* off-target ❌ |

Underfitting is the bottom-left. Overfitting is the top-right.

In [ ]:
# Visualise the four combinations
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
configs = [("Low bias\nLow variance", 0.0, 0.10),
           ("Low bias\nHigh variance", 0.0, 0.45),
           ("High bias\nLow variance", 0.55, 0.10),
           ("High bias\nHigh variance", 0.55, 0.45)]
for ax, (title, bias, sd) in zip(axes, configs):
    ang = rng.uniform(0, 2*np.pi, 30)
    px = bias * np.cos(0.7) + rng.normal(0, sd, 30)
    py = bias * np.sin(0.7) + rng.normal(0, sd, 30)
    for r in (1.0, 0.66, 0.33):
        ax.add_patch(plt.Circle((0, 0), r, fill=False, color="grey", lw=1))
    ax.add_patch(plt.Circle((0, 0), 0.1, color="crimson", alpha=0.5))
    ax.scatter(px, py, s=28, color="steelblue", edgecolor="k", linewidth=0.3)
    ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3); ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    mse = (px**2 + py**2).mean()
    ax.set_title(f"{title}\nmean squared distance = {mse:.3f}", fontsize=9)
plt.tight_layout(); plt.show()

---
## 12.2 Measuring bias and variance by simulation

In real life you cannot compute bias, because you do not know $f$. But in a simulation you
do — so this is where the concepts stop being abstract.

The procedure:

1. Fix a true function $f$ and a noise level $\sigma$
2. Draw **many** independent training sets from it
3. Fit the model on each one, and predict at a fixed grid of test points
4. At each test point:
   - $\text{Bias}^2 = (\overline{\hat{f}(x)} - f(x))^2$, where the bar averages over training sets
   - $\text{Variance} = $ variance of $\hat{f}(x)$ across training sets
5. Average over the grid, and add $\sigma^2$

In [ ]:
def f_true(x):
    return np.sin(1.5 * x) + 0.3 * x

SIGMA = 0.4
N_TRAIN = 40
N_SETS = 300
x_grid = np.linspace(0.2, 6.8, 120)

def bias_variance(make_model, n_sets=N_SETS, n_train=N_TRAIN, seed=0):
    '''Estimate bias^2, variance and total error of a model on the fixed grid.'''
    g = np.random.default_rng(seed)
    preds = np.empty((n_sets, len(x_grid)))
    for i in range(n_sets):
        xs = g.uniform(0.2, 6.8, n_train)
        ys = f_true(xs) + g.normal(0, SIGMA, n_train)
        mdl = make_model().fit(xs.reshape(-1, 1), ys)
        preds[i] = mdl.predict(x_grid.reshape(-1, 1))
    mean_pred = preds.mean(axis=0)
    bias2 = ((mean_pred - f_true(x_grid)) ** 2).mean()
    variance = preds.var(axis=0).mean()
    return {"bias2": bias2, "variance": variance, "noise": SIGMA**2,
            "total": bias2 + variance + SIGMA**2}, preds, mean_pred

res_simple, preds_simple, mean_simple = bias_variance(
    lambda: make_pipeline(PolynomialFeatures(1), LinearRegression()), seed=1)
res_good, preds_good, mean_good = bias_variance(
    lambda: make_pipeline(PolynomialFeatures(5), LinearRegression()), seed=1)
res_complex, preds_complex, mean_complex = bias_variance(
    lambda: make_pipeline(PolynomialFeatures(14), LinearRegression()), seed=1)

table = pd.DataFrame([
    {"model": "degree 1 (too simple)", **res_simple},
    {"model": "degree 5 (about right)", **res_good},
    {"model": "degree 14 (too complex)", **res_complex},
])
print(table.round(4).to_string(index=False))
print(f"\nIrreducible error sigma^2 = {SIGMA**2:.4f} in every row -- it is a property of the")
print("problem, not of the model.")

In [ ]:
# Draw the 300 fitted curves for each model. This picture IS the decomposition.
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
sets = [("degree 1: biased, stable", preds_simple, mean_simple, res_simple),
        ("degree 5: balanced", preds_good, mean_good, res_good),
        ("degree 14: unbiased on average, wild", preds_complex, mean_complex, res_complex)]
for ax, (title, preds, mean_pred, res) in zip(axes, sets):
    for i in range(0, 120, 2):
        ax.plot(x_grid, preds[i], color="steelblue", alpha=0.08, lw=1)
    ax.plot(x_grid, f_true(x_grid), "k--", lw=2, label="true f(x)")
    ax.plot(x_grid, mean_pred, color="crimson", lw=2.2, label="average of 300 fits")
    ax.set_ylim(-3, 4); ax.legend(fontsize=7)
    ax.set_title(f"{title}\nbias^2={res['bias2']:.3f}  var={res['variance']:.3f}", fontsize=9)
plt.tight_layout(); plt.show()

print("Left  : every fit looks the same (low variance) but none matches the curve (high bias)")
print("Middle: the average tracks the truth AND individual fits stay close to it")
print("Right : the average is nearly right, but any single fit can be far off (high variance)")
print("\nHigh capacity does not mean high bias -- it means the AVERAGE is right and the")
print("individual is unreliable. That distinction is the whole point.")

In [ ]:
# Where does the error live along x? Bias and variance are not uniform.
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for name, preds, colour in [("degree 1", preds_simple, "steelblue"),
                            ("degree 5", preds_good, "seagreen"),
                            ("degree 14", preds_complex, "crimson")]:
    ax[0].plot(x_grid, (preds.mean(0) - f_true(x_grid))**2, color=colour, lw=2, label=name)
    ax[1].plot(x_grid, preds.var(0), color=colour, lw=2, label=name)
ax[0].set_title("Bias^2 across the input range"); ax[0].set_yscale("log")
ax[1].set_title("Variance across the input range"); ax[1].set_yscale("log")
for a_ in ax:
    a_.set_xlabel("x"); a_.legend(fontsize=8)
plt.tight_layout(); plt.show()

print("Variance explodes at the EDGES of the data range for the flexible model.")
print("That is the mathematical reason extrapolation is dangerous (Notebook 9).")

---
## 12.3 The trade-off curve

Sweep complexity and plot all three terms together. The classic U-shape of total error is
the sum of a falling bias curve and a rising variance curve.

In [ ]:
degrees = range(1, 16)
rows = []
for d in degrees:
    r, _, _ = bias_variance(lambda d=d: make_pipeline(PolynomialFeatures(d),
                                                      LinearRegression()), seed=2)
    rows.append({"degree": d, **r})
bv = pd.DataFrame(rows)

plt.plot(bv.degree, bv.bias2, "o-", color="steelblue", label="bias$^2$")
plt.plot(bv.degree, bv.variance, "o-", color="darkorange", label="variance")
plt.axhline(SIGMA**2, color="grey", ls=":", label=f"irreducible = {SIGMA**2:.2f}")
plt.plot(bv.degree, bv.total, "o-", color="crimson", lw=2.4, label="total expected error")
best = int(bv.loc[bv.total.idxmin(), "degree"])
plt.axvline(best, color="black", ls="--", label=f"optimum = degree {best}")
plt.yscale("log"); plt.xlabel("model complexity (polynomial degree)")
plt.ylabel("expected squared error (log scale)")
plt.title("The bias-variance tradeoff")
plt.legend(fontsize=8); plt.show()

print(bv.round(4).to_string(index=False))
print(f"\nOptimal complexity: degree {best}")
print("Left of the optimum you are bias-limited; right of it you are variance-limited.")

In [ ]:
# Two more knobs, same story. k-NN: capacity falls as k RISES.
ks = [1, 2, 3, 5, 8, 12, 20, 30, 40]
rows_k = []
for k in ks:
    r, _, _ = bias_variance(lambda k=k: KNeighborsRegressor(n_neighbors=k), seed=3)
    rows_k.append({"k": k, **r})
bk = pd.DataFrame(rows_k)

# Ridge: capacity falls as alpha RISES
alphas = [1e-6, 1e-4, 1e-2, 0.1, 1, 10, 100, 1000]
rows_a = []
for al in alphas:
    r, _, _ = bias_variance(lambda al=al: make_pipeline(PolynomialFeatures(12),
                                                        StandardScaler(),
                                                        Ridge(alpha=al)), seed=4)
    rows_a.append({"alpha": al, **r})
ba = pd.DataFrame(rows_a)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(bk.k, bk.bias2, "o-", color="steelblue", label="bias$^2$")
ax[0].plot(bk.k, bk.variance, "o-", color="darkorange", label="variance")
ax[0].plot(bk.k, bk.total, "o-", color="crimson", lw=2.2, label="total")
ax[0].set_xlabel("k (larger k = simpler model)"); ax[0].set_yscale("log")
ax[0].set_title("k-NN"); ax[0].legend(fontsize=8)

ax[1].plot(ba.alpha, ba.bias2, "o-", color="steelblue", label="bias$^2$")
ax[1].plot(ba.alpha, ba.variance, "o-", color="darkorange", label="variance")
ax[1].plot(ba.alpha, ba.total, "o-", color="crimson", lw=2.2, label="total")
ax[1].set_xscale("log"); ax[1].set_yscale("log")
ax[1].set_xlabel("ridge alpha (larger = simpler model)")
ax[1].set_title("Degree-12 polynomial with ridge penalty"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"k-NN  optimum: k = {int(bk.loc[bk.total.idxmin(), 'k'])}")
print(f"Ridge optimum: alpha = {ba.loc[ba.total.idxmin(), 'alpha']}")
print("\nRidge is the clearest illustration of the trade: turning alpha up deliberately")
print("ADDS bias in order to REMOVE more variance than it adds. That is a good deal")
print("whenever variance dominates.")

---
## 12.4 Irreducible error: the floor

$\sigma^2$ is the variance of $y$ given $x$ — the part of the outcome that your features
simply do not determine. Two people with identical recorded features can have different
outcomes, and no algorithm fixes that.

Practical consequences:

- If your RMSE is close to $\sigma$, **stop tuning**. Buy better features instead.
- If a reported error is *below* $\sigma$, suspect leakage.
- "Better features" and "more data" attack different terms: better features lower $\sigma^2$
  **and** bias; more rows lower variance.

In [ ]:
print("Same model, same n, different noise levels:")
print(f"{'sigma':>8}{'sigma^2':>10}{'bias^2':>10}{'variance':>10}{'total':>10}{'% irreducible':>15}")
for s_ in (0.1, 0.2, 0.4, 0.8, 1.6):
    saved = SIGMA
    globals()["SIGMA"] = s_
    r, _, _ = bias_variance(lambda: make_pipeline(PolynomialFeatures(5), LinearRegression()),
                            n_sets=150, seed=5)
    globals()["SIGMA"] = saved
    print(f"{s_:>8.2f}{s_**2:>10.4f}{r['bias2']:>10.4f}{r['variance']:>10.4f}"
          f"{r['total']:>10.4f}{s_**2/r['total']*100:>14.1f}%")

print("\nAt sigma = 1.6 the noise is 90% of the total error. Tuning the model there is")
print("almost pointless -- the leverage is in measurement quality and new features.")

In [ ]:
# More data attacks variance, not bias
print("Degree-12 polynomial, increasing training size:")
print(f"{'n_train':>9}{'bias^2':>10}{'variance':>10}{'total':>10}")
for nt in (20, 40, 80, 200, 600):
    r, _, _ = bias_variance(lambda: make_pipeline(PolynomialFeatures(12), LinearRegression()),
                            n_sets=150, n_train=nt, seed=6)
    print(f"{nt:>9}{r['bias2']:>10.4f}{r['variance']:>10.4f}{r['total']:>10.4f}")

print("\nBias barely moves (it is a property of the model class). Variance collapses.")
print("This is exactly why the learning curves in Notebook 11 converge -- and why 'more")
print("data' cures overfitting but never cures underfitting.")

---
## 12.5 Where models sit on the spectrum

| Model | Typical bias | Typical variance | Notes |
|---|---|---|---|
| Predict the mean | very high | zero | the baseline every model must beat |
| Linear / logistic regression | high | low | strong assumptions, very stable |
| Ridge / Lasso | higher than OLS | lower than OLS | you choose the point on the curve |
| Naive Bayes | high | low | independence assumption is a bias |
| Shallow decision tree | high | moderate | |
| Deep decision tree | low | **very high** | the canonical high-variance model |
| k-NN, small $k$ | low | very high | |
| k-NN, large $k$ | high | low | |
| SVM (RBF, large $C,\gamma$) | low | high | |
| **Random forest / bagging** | ≈ single tree | **much lower** | variance reduction by averaging |
| **Gradient boosting** | **lower** than a tree | moderate–high | bias reduction by sequential correction |
| Deep neural network | very low | high (managed by regularisation and scale) | |

Two families, two strategies:

- **Bagging** (random forests): take many low-bias, high-variance models and *average* them.
  Bias stays put, variance falls by roughly $1/B$ for $B$ independent models — less in
  practice, because trees on similar data are correlated. Random feature selection exists to
  decorrelate them.
- **Boosting**: take a high-bias weak learner and fit each new one to the previous errors.
  Bias falls steadily; variance creeps up, so you need early stopping.

In [ ]:
# Bagging: same bias, far less variance
single = bias_variance(lambda: DecisionTreeRegressor(random_state=0), n_sets=150, seed=7)[0]
bag10  = bias_variance(lambda: BaggingRegressor(DecisionTreeRegressor(),
                                                n_estimators=10, random_state=0),
                       n_sets=150, seed=7)[0]
bag100 = bias_variance(lambda: RandomForestRegressor(n_estimators=100, random_state=0),
                       n_sets=150, seed=7)[0]
boost  = bias_variance(lambda: GradientBoostingRegressor(n_estimators=120, max_depth=2,
                                                         learning_rate=0.1, random_state=0),
                       n_sets=150, seed=7)[0]
stump  = bias_variance(lambda: DecisionTreeRegressor(max_depth=1, random_state=0),
                       n_sets=150, seed=7)[0]

comp = pd.DataFrame([
    {"model": "single unpruned tree", **single},
    {"model": "bagging, 10 trees", **bag10},
    {"model": "random forest, 100 trees", **bag100},
    {"model": "decision stump (depth 1)", **stump},
    {"model": "gradient boosting of stumps", **boost},
])
print(comp.round(4).to_string(index=False))
print()
print(f"Bagging: bias^2 {single['bias2']:.4f} -> {bag100['bias2']:.4f} (barely changed),")
print(f"         variance {single['variance']:.4f} -> {bag100['variance']:.4f} "
      f"({(1-bag100['variance']/single['variance'])*100:.0f}% reduction)")
print(f"Boosting: stump bias^2 {stump['bias2']:.4f} -> {boost['bias2']:.4f} "
      f"({(1-boost['bias2']/stump['bias2'])*100:.0f}% reduction)")
print("\nTwo different medicines for two different diseases.")

In [ ]:
# How variance falls with the number of bagged trees
counts = [1, 2, 5, 10, 25, 50, 100, 200]
vars_ = []
for B in counts:
    r, _, _ = bias_variance(lambda B=B: RandomForestRegressor(n_estimators=B, random_state=0),
                            n_sets=80, seed=8)
    vars_.append(r["variance"])

plt.plot(counts, vars_, "o-", color="steelblue", label="measured variance")
plt.plot(counts, vars_[0] / np.array(counts), "k--", lw=1.2,
         label="1/B (if trees were independent)")
plt.xscale("log"); plt.yscale("log")
plt.xlabel("number of trees B"); plt.ylabel("variance")
plt.title("Averaging reduces variance, but the trees are correlated so it plateaus")
plt.legend(fontsize=8); plt.show()

print("The gap between the two curves is the correlation between trees. Random forests")
print("subsample FEATURES at each split precisely to push the solid curve toward the dashed")
print("one -- decorrelating the ensemble is worth more than making each tree better.")

---
## 12.6 The classification version

For 0–1 loss the decomposition is not as clean — the algebra of squared error does not carry
over — but the *concepts* do, and you can measure them directly:

- **Bias-like term:** how often does the *majority vote* across training sets get it wrong?
- **Variance-like term:** how often do models trained on different samples *disagree*?

A useful practical consequence: classifiers tolerate more variance than regressors, because
a noisy prediction only hurts if it crosses the decision boundary.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

Xtest_c, ytest_c = make_moons(n_samples=600, noise=0.3, random_state=99)

def classification_bias_variance(make_model, n_sets=120, n_train=120, seed=0):
    '''Main-prediction error ('bias') and disagreement ('variance') for a classifier.'''
    g = np.random.default_rng(seed)
    preds = np.empty((n_sets, len(ytest_c)), dtype=int)
    for i in range(n_sets):
        Xs, ys = make_moons(n_samples=n_train, noise=0.3, random_state=int(g.integers(0, 10**6)))
        preds[i] = make_model().fit(Xs, ys).predict(Xtest_c)
    majority = (preds.mean(axis=0) > 0.5).astype(int)
    bias_like = (majority != ytest_c).mean()
    variance_like = (preds != majority).mean()
    avg_error = (preds != ytest_c).mean()
    return {"bias_like": bias_like, "variance_like": variance_like, "avg_error": avg_error}

models = {
    "logistic regression":       lambda: LogisticRegression(),
    "k-NN, k=1":                 lambda: KNeighborsClassifier(n_neighbors=1),
    "k-NN, k=15":                lambda: KNeighborsClassifier(n_neighbors=15),
    "deep decision tree":        lambda: DecisionTreeClassifier(random_state=0),
    "pruned tree (depth 3)":     lambda: DecisionTreeClassifier(max_depth=3, random_state=0),
    "random forest, 100 trees":  lambda: RandomForestClassifier(n_estimators=100, random_state=0),
}
rows_c = [{"model": name, **classification_bias_variance(f, seed=9)} for name, f in models.items()]
cdf = pd.DataFrame(rows_c)
print(cdf.round(4).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
idx = np.arange(len(cdf))
ax.barh(idx, cdf.bias_like, height=0.4, label="bias-like (majority vote is wrong)",
        color="steelblue")
ax.barh(idx + 0.42, cdf.variance_like, height=0.4,
        label="variance-like (models disagree)", color="darkorange")
ax.set_yticks(idx + 0.21); ax.set_yticklabels(cdf.model)
ax.set_xlabel("rate"); ax.legend(fontsize=8)
ax.set_title("Bias-like and variance-like error by classifier")
plt.tight_layout(); plt.show()

print("Reading the chart:")
print("  logistic regression : low disagreement, high majority error -> high bias")
print("  k-NN k=1, deep tree : low majority error, high disagreement -> high variance")
print("  random forest       : keeps the low bias of trees, cuts the disagreement")
print("\nThe forest is the best of both -- which is why it is such a strong default.")

---
## 12.7 What each remedy actually does

| Action | Bias | Variance | Irreducible | When to use it |
|---|---|---|---|---|
| More training rows | — | **↓↓** | — | high variance |
| More / better features | **↓** | ↑ | **↓** | high bias, or a high noise floor |
| Remove features | ↑ | ↓ | — | high variance, many weak predictors |
| Increase model capacity | **↓** | ↑ | — | high bias |
| Regularisation (ridge/lasso) | ↑ | **↓** | — | high variance |
| Bagging / random forest | — | **↓↓** | — | high variance |
| Boosting | **↓↓** | ↑ | — | high bias |
| Early stopping | ↑ | ↓ | — | high variance in iterative models |
| Cross-validation | — | — | — | it *measures*; it does not change the model |
| Better measurement instruments | ↓ | — | **↓** | when $\sigma^2$ dominates |

The last row is the one data scientists forget. If the noise floor is the problem, the fix
is not in the model at all.

In [ ]:
# Prove the two most common remedies on one high-variance starting point
start = lambda: make_pipeline(PolynomialFeatures(13), LinearRegression())
base_r, _, _ = bias_variance(start, n_sets=200, seed=10)

remedies = {
    "baseline (degree 13, n=40)":       (start, 40),
    "+ regularisation (ridge a=1)":     (lambda: make_pipeline(PolynomialFeatures(13),
                                                               StandardScaler(),
                                                               Ridge(alpha=1.0)), 40),
    "+ less capacity (degree 4)":       (lambda: make_pipeline(PolynomialFeatures(4),
                                                               LinearRegression()), 40),
    "+ more data (n=400, degree 13)":   (start, 400),
    "+ bagging the degree-13 fit":      (lambda: BaggingRegressor(
                                            make_pipeline(PolynomialFeatures(13),
                                                          LinearRegression()),
                                            n_estimators=30, random_state=0), 40),
}
out = []
for name, (mk, nt) in remedies.items():
    r, _, _ = bias_variance(mk, n_sets=200, n_train=nt, seed=10)
    out.append({"treatment": name, **r})
res_df = pd.DataFrame(out)
res_df["total_vs_baseline"] = (res_df.total / base_r["total"] - 1).round(3)
print(res_df.round(4).to_string(index=False))
print("\nAll four treatments cut the total error, and the columns show they do it")
print("differently: regularisation and bagging trade a little bias for a lot of variance,")
print("reducing capacity trades more bias for more variance reduction, and more data")
print("removes variance for free.")

---
## 12.8 Where the classical picture breaks: double descent

The U-shaped curve is the classical story, and it is correct for the models in this course.
But since about 2019 it has been clear that it is **incomplete**.

Push complexity past the **interpolation threshold** — the point where the model has just
enough parameters to fit the training data exactly — and test error can *fall again*:

```
test error
    |    classical U            modern regime
    |   /\
    |  /  \        /\
    | /    \      /  \___________
    |/      \____/                 <- second descent
    +------------------------------> model capacity
              ^ interpolation threshold (parameters = n)
```

The intuition: among the many parameter settings that interpolate the training data,
gradient descent and ridge-like penalties prefer *smooth* ones, and smooth interpolators
generalise. This is a large part of why enormous neural networks work at all.

What this does **not** mean: that you should stop regularising, or that the trade-off is
fake. In the small-data, tabular, moderate-capacity setting you will work in for this
course, the classical U-shape is what you will observe and what you should design for.

In [ ]:
# A minimal double-descent demonstration: random-feature regression
def double_descent(n_train=60, n_test=800, noise=0.6, seed=0):
    g = np.random.default_rng(seed)
    d_in = 20
    W = g.normal(size=(d_in, 2000))                       # a fixed random feature map
    beta = g.normal(size=d_in)

    Xtr = g.normal(size=(n_train, d_in))
    Xte = g.normal(size=(n_test, d_in))
    ytr = Xtr @ beta + g.normal(0, noise, n_train)
    yte = Xte @ beta + g.normal(0, noise, n_test)

    def feats(X, p):
        return np.tanh(X @ W[:, :p])

    ps, test_rmse, train_rmse = [], [], []
    for p in [2, 5, 10, 20, 30, 40, 50, 55, 58, 60, 62, 65, 70, 80, 120, 200, 400, 800, 1600]:
        Ftr, Fte = feats(Xtr, p), feats(Xte, p)
        # minimum-norm least squares (what gradient descent converges to)
        coef, *_ = np.linalg.lstsq(Ftr, ytr, rcond=None)
        ps.append(p)
        train_rmse.append(np.sqrt(mean_squared_error(ytr, Ftr @ coef)))
        test_rmse.append(np.sqrt(mean_squared_error(yte, Fte @ coef)))
    return np.array(ps), np.array(train_rmse), np.array(test_rmse)

ps, tr_dd, te_dd = double_descent(seed=3)

plt.plot(ps, tr_dd, "o-", color="steelblue", label="training RMSE")
plt.plot(ps, te_dd, "o-", color="crimson", label="test RMSE")
plt.axvline(60, color="black", ls="--", label="interpolation threshold (p = n = 60)")
plt.xscale("log"); plt.yscale("log")
plt.xlabel("number of random features p"); plt.ylabel("RMSE (log scale)")
plt.title("Double descent: error spikes at p = n, then falls again")
plt.legend(fontsize=8); plt.show()

peak = ps[int(np.argmax(te_dd))]
print(f"Test error peaks at p = {peak} (n = 60), then improves as p grows past it.")
print(f"  test RMSE at p=40   : {te_dd[ps == 40][0]:.3f}")
print(f"  test RMSE at p=60   : {te_dd[ps == 60][0]:.3f}   <- the spike")
print(f"  test RMSE at p=1600 : {te_dd[ps == 1600][0]:.3f}   <- better than either")
print("\nThe classical U is the left half of this picture. Both halves are real.")

---
## Exercises

**Exercise 1.** Estimate bias², variance and total error for a decision tree at depths
1, 2, 3, 5, 8 and unlimited on the simulated function. Identify the depth that minimises
total error, and say which term dominates at each end.

In [ ]:
# --- Solution 1 -------------------------------------------------------------
depths = [1, 2, 3, 5, 8, None]
rows1 = []
for d in depths:
    r, _, _ = bias_variance(lambda d=d: DecisionTreeRegressor(max_depth=d, random_state=0),
                            n_sets=200, seed=21)
    rows1.append({"max_depth": str(d), **r,
                  "dominant": "bias" if r["bias2"] > r["variance"] else "variance"})
t1 = pd.DataFrame(rows1)
print(t1.round(4).to_string(index=False))

best_row = t1.loc[t1.total.idxmin()]
print(f"\nLowest total error at max_depth = {best_row['max_depth']} "
      f"(total {best_row['total']:.4f})")
print("Shallow end: bias dominates -- the tree cannot bend enough to follow the sine wave.")
print("Deep end   : variance dominates -- each tree chases its own sample's noise.")

plt.plot(range(len(t1)), t1.bias2, "o-", color="steelblue", label="bias^2")
plt.plot(range(len(t1)), t1.variance, "o-", color="darkorange", label="variance")
plt.plot(range(len(t1)), t1.total, "o-", color="crimson", lw=2.2, label="total")
plt.xticks(range(len(t1)), t1.max_depth); plt.xlabel("max_depth"); plt.yscale("log")
plt.legend(fontsize=8); plt.title("Decision-tree bias-variance tradeoff"); plt.show()

**Exercise 2.** A colleague says "I will fix my overfitting by adding more features".
Use the decomposition to show why that is usually wrong, and identify the one case where it
is right.

In [ ]:
# --- Solution 2 -------------------------------------------------------------
# Case A: overfitting already, and the extra features are noise
def with_noise_features(p_noise, n_train=50, n_sets=150, seed=22):
    g = np.random.default_rng(seed)
    preds = np.empty((n_sets, 200))
    Xte = g.normal(size=(200, 1 + p_noise))
    f_te = 2.0 * Xte[:, 0] + 0.8 * Xte[:, 0]**2
    for i in range(n_sets):
        Xs = g.normal(size=(n_train, 1 + p_noise))
        ys = 2.0*Xs[:, 0] + 0.8*Xs[:, 0]**2 + g.normal(0, 1.0, n_train)
        preds[i] = LinearRegression().fit(Xs, ys).predict(Xte)
    return {"bias2": ((preds.mean(0) - f_te)**2).mean(),
            "variance": preds.var(0).mean()}

print("Adding NOISE features to an already-overfitting model:")
print(f"{'extra noise features':>22}{'bias^2':>10}{'variance':>11}{'sum':>10}")
for p_noise in (0, 5, 15, 30, 45):
    r = with_noise_features(p_noise)
    print(f"{p_noise:>22}{r['bias2']:>10.4f}{r['variance']:>11.4f}"
          f"{r['bias2']+r['variance']:>10.4f}")
print("\nVariance climbs steeply. Every irrelevant column is another coefficient estimated")
print("from the same 50 rows, so the model gets shakier while bias barely improves.")

In [ ]:
# Case B: the one situation where more features IS the fix -- when you are UNDERFITTING
# and the new feature carries real signal (here: the missing squared term).
def underfit_case(include_square, n_train=50, n_sets=150, seed=23):
    g = np.random.default_rng(seed)
    Xte = g.normal(size=(300, 1))
    f_te = 2.0*Xte[:, 0] + 0.8*Xte[:, 0]**2
    preds = np.empty((n_sets, 300))
    for i in range(n_sets):
        Xs = g.normal(size=(n_train, 1))
        ys = 2.0*Xs[:, 0] + 0.8*Xs[:, 0]**2 + g.normal(0, 1.0, n_train)
        A = np.column_stack([Xs, Xs**2]) if include_square else Xs
        B = np.column_stack([Xte, Xte**2]) if include_square else Xte
        preds[i] = LinearRegression().fit(A, ys).predict(B)
    return {"bias2": ((preds.mean(0) - f_te)**2).mean(), "variance": preds.var(0).mean()}

no_sq = underfit_case(False)
with_sq = underfit_case(True)
print("Adding a feature that carries REAL signal (x^2), to an underfitting model:")
print(f"  linear only   : bias^2 {no_sq['bias2']:.4f}  variance {no_sq['variance']:.4f}  "
      f"sum {no_sq['bias2']+no_sq['variance']:.4f}")
print(f"  with x^2 term : bias^2 {with_sq['bias2']:.4f}  variance {with_sq['variance']:.4f}  "
      f"sum {with_sq['bias2']+with_sq['variance']:.4f}")
print()
print("Conclusion for the colleague:")
print("  Adding features attacks BIAS and inflates VARIANCE. If you are overfitting,")
print("  variance is already your problem, so you are pouring fuel on the fire.")
print("  The exception: a feature that is genuinely informative and currently missing --")
print("  then bias falls far more than variance rises. Diagnose first (learning curve),")
print("  and check that the new feature carries signal before adding it.")

**Exercise 3.** Show experimentally that bagging reduces variance roughly in proportion to
how *uncorrelated* the base models are. Compare bagging a high-variance model (deep tree)
with bagging a low-variance one (linear regression), and explain the difference.

In [ ]:
# --- Solution 3 -------------------------------------------------------------
pairs = [
    ("deep tree",        lambda: DecisionTreeRegressor(random_state=0),
     lambda: BaggingRegressor(DecisionTreeRegressor(), n_estimators=50, random_state=0)),
    ("linear regression", lambda: LinearRegression(),
     lambda: BaggingRegressor(LinearRegression(), n_estimators=50, random_state=0)),
    ("k-NN, k=1",        lambda: KNeighborsRegressor(n_neighbors=1),
     lambda: BaggingRegressor(KNeighborsRegressor(n_neighbors=1), n_estimators=50,
                              random_state=0)),
]
rows3 = []
for name, base_f, bag_f in pairs:
    rb, _, _ = bias_variance(base_f, n_sets=150, seed=24)
    rg, _, _ = bias_variance(bag_f, n_sets=150, seed=24)
    rows3.append({"base model": name,
                  "base bias^2": round(rb["bias2"], 4), "base var": round(rb["variance"], 4),
                  "bagged bias^2": round(rg["bias2"], 4), "bagged var": round(rg["variance"], 4),
                  "var reduction": f"{(1 - rg['variance']/rb['variance'])*100:.0f}%",
                  "total change": f"{(rg['total']/rb['total'] - 1)*100:+.0f}%"})
print(pd.DataFrame(rows3).to_string(index=False))
print()
print("Explanation:")
print("  Bagging averages models fitted to bootstrap resamples. The variance of an average")
print("  of B models with pairwise correlation rho is rho*V + (1-rho)*V/B.")
print("  * A deep tree is UNSTABLE: resampling changes its splits, so rho is small and")
print("    averaging cancels a lot of variance.")
print("  * Linear regression is STABLE: every bootstrap gives nearly the same line, so")
print("    rho is near 1 and averaging achieves almost nothing.")
print("  Bagging is a variance treatment. Prescribe it only for high-variance patients.")

**Exercise 4 (challenge).** Verify the decomposition **numerically**: show that for a fixed
test point $x_0$, the average squared error over many training sets equals
$\text{bias}^2 + \text{variance} + \sigma^2$ to within simulation error. Then explain why the
identity requires squared error and does not hold for MAE.

In [ ]:
# --- Solution 4 -------------------------------------------------------------
x0 = 3.7
f0 = f_true(x0)
SIG = 0.4
n_sets, n_train = 6000, 40

g = np.random.default_rng(77)
fhat = np.empty(n_sets)
sq_err = np.empty(n_sets)
abs_err = np.empty(n_sets)

for i in range(n_sets):
    xs = g.uniform(0.2, 6.8, n_train)
    ys = f_true(xs) + g.normal(0, SIG, n_train)
    mdl = make_pipeline(PolynomialFeatures(6), LinearRegression()).fit(xs.reshape(-1, 1), ys)
    fhat[i] = mdl.predict([[x0]])[0]
    y0 = f0 + g.normal(0, SIG)                  # a fresh noisy observation at x0
    sq_err[i] = (y0 - fhat[i]) ** 2
    abs_err[i] = abs(y0 - fhat[i])

bias = fhat.mean() - f0
bias2 = bias ** 2
var = fhat.var()
noise = SIG ** 2

print(f"At x0 = {x0}:  f(x0) = {f0:.4f},  average prediction = {fhat.mean():.4f}\n")
print(f"  bias^2                        = {bias2:.5f}")
print(f"  variance of predictions       = {var:.5f}")
print(f"  irreducible sigma^2           = {noise:.5f}")
print(f"  --------------------------------------------")
print(f"  sum of the three terms        = {bias2 + var + noise:.5f}")
print(f"  measured mean squared error   = {sq_err.mean():.5f}")
print(f"  discrepancy                   = {abs(sq_err.mean() - (bias2+var+noise)):.5f}"
      f"  (Monte-Carlo error, ~1/sqrt({n_sets}))")

print(f"\nFor MAE there is no such identity:")
print(f"  |bias| + sd(fhat) + sigma   = {abs(bias) + fhat.std() + SIG:.5f}")
print(f"  measured mean absolute error = {abs_err.mean():.5f}   <- does NOT match")
print()
print("Why squared error is special:")
print("  Squared error is a quadratic form, so E[(A+B)^2] = E[A^2] + E[B^2] whenever")
print("  A and B are independent with mean zero. The cross terms vanish, which is exactly")
print("  what lets the error split cleanly into three additive pieces.")
print("  Absolute value is not quadratic; |A+B| != |A| + |B| in general, so the cross terms")
print("  survive and no clean decomposition exists. The CONCEPTS of bias and variance")
print("  still apply to any loss -- only the exact three-way arithmetic is specific to MSE.")

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.hist(fhat, bins=60, color="steelblue", edgecolor="none", label="predictions at x0")
ax.axvline(f0, color="black", lw=2, ls="--", label=f"true f(x0) = {f0:.3f}")
ax.axvline(fhat.mean(), color="crimson", lw=2, label=f"mean prediction = {fhat.mean():.3f}")
ax.set_xlabel("prediction at x0"); ax.set_title("Spread = variance; distance of red from black = bias")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

---
## Summary

| Concept | Formula / key point |
|---|---|
| Decomposition | $\mathbb{E}[(y-\hat{f})^2] = \text{Bias}^2 + \text{Variance} + \sigma^2$ |
| Bias | $\mathbb{E}[\hat{f}(x)] - f(x)$ — error of the *average* model |
| Variance | Spread of $\hat{f}(x)$ across training sets |
| Irreducible error | $\sigma^2$ — the floor; attack it with better features/measurement |
| Underfitting | Bias-dominated |
| Overfitting | Variance-dominated |
| More data | Cuts variance, leaves bias alone |
| More features | Cuts bias, raises variance (unless they are noise, then only the latter) |
| Regularisation | Buys bias to sell more variance |
| Bagging / forests | Variance reduction; works best on unstable base models |
| Boosting | Bias reduction; needs early stopping |
| Only squared error decomposes exactly | The cross terms vanish because the loss is quadratic |
| Double descent | Past the interpolation threshold the classical U no longer applies |

---

## 🎓 Module complete

**Statistical Foundations for Data Science** — twelve notebooks:

| | Notebook | Theme |
|---|---|---|
| 1 | [Probability Basics](1.%20Probability%20Basics.ipynb) | The language of uncertainty |
| 2 | [Random Variables](2.%20Random%20Variables.ipynb) | Attaching numbers to outcomes |
| 3 | [Probability Distributions](3.%20Probability%20Distributions.ipynb) | Templates for randomness |
| 4 | [Sampling Techniques](4.%20Sampling%20Techniques.ipynb) | From sample to population |
| 5 | [Correlation and Covariance](5.%20Correlation%20and%20Covariance.ipynb) | Relationships between variables |
| 6 | [Hypothesis Testing](6.%20Hypothesis%20Testing.ipynb) | Is this real? |
| 7 | [t-Test](7.%20t-Test.ipynb) | Comparing means |
| 8 | [Chi-Square Test](8.%20Chi-Square%20Test.ipynb) | Comparing counts |
| 9 | [Regression Basics](9.%20Regression%20Basics.ipynb) | Fitting and interpreting equations |
| 10 | [Train-Test Split](10.%20Train-Test%20Split.ipynb) | Honest measurement |
| 11 | [Overfitting](11.%20Overfitting.ipynb) | Diagnosis and treatment |
| 12 | [Bias-Variance Tradeoff](12.%20Bias-Variance%20Tradeoff.ipynb) | Why the trade-off exists |

**Next module:** [7. Machine Learning Fundamentals and Predictive Analytics](../7.%20Machine%20Learning%20Fundamentals%20and%20Predictive%20Analytics),
which applies all of this to the standard algorithm toolkit.